In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
import pandas as pd
import json

In [7]:
train = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/train.csv")
test = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

train.head(2)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


In [11]:
row = train.iloc[0]

prompt_list = json.loads(row["prompt"])

print("type before:", type(row["prompt"]))
print("type after: ", type(prompt_list))
print("number of items:", len(prompt_list))
print()
print("Item 0:", prompt_list[0])
print()
print("Item 1:", prompt_list[1])

type before: <class 'str'>
type after:  <class 'list'>
number of items: 2

Item 0: Is it morally right to try to have a certain percentage of females on managerial positions?

Item 1: OK, does pineapple belong on a pizza? Relax and give me fun answer.


In [13]:
resp_a = json.loads(row["response_a"])

print("number of items in response_a:", len(resp_a))
print(resp_a[0][:800])

number of items in response_a: 2
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.

2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.

3. 


In [14]:
def clean_text(s):
    items = json.loads(s)                          # text -> real list
    items = [x for x in items if x is not None]    # throw away missing turns
    return "\n".join(items)                        # glue the turns into one string


train["prompt_text"] = train["prompt"].apply(clean_text)
train["response_a_text"] = train["response_a"].apply(clean_text)
train["response_b_text"] = train["response_b"].apply(clean_text)

In [18]:
print(train.shape)
print(train[["prompt_text", "response_a_text", "response_b_text"]].head(3))

(57477, 12)
                                         prompt_text  \
0  Is it morally right to try to have a certain p...   
1  What is the difference between marriage licens...   
2  explain function calling. how would you call a...   

                                     response_a_text  \
0  The question of whether it is morally right to...   
1  A marriage license is a legal document that al...   
2  Function calling is the process of invoking or...   

                                     response_b_text  
0  As an AI, I don't have personal beliefs or opi...  
1  A marriage license and a marriage certificate ...  
2  Function calling is the process of invoking a ...  


In [20]:
label_cols = ["winner_model_a", "winner_model_b", "winner_tie"]

train["label"] = train[label_cols].values.argmax(axis=1)

print(train[label_cols + ["label"]].head(5))

   winner_model_a  winner_model_b  winner_tie  label
0               1               0           0      0
1               0               1           0      1
2               0               0           1      2
3               1               0           0      0
4               0               1           0      1


In [21]:
print((train[label_cols].sum(axis=1) == 1).all())

True
